# Versuch IL: Ionenleitung

| | |
|---|---|
| **Name** | *Vorname Nachname* |
| **Matrikelnummer** | *123456* |
| **Gruppe** | *…* |
| **Betreuer:in** | *…* |
| **Versuchstag** | *TT.MM.JJJJ* |

**Vorübergehend:** Im Praktikumsprofil fehlen derzeit noch Pakete, die dieses Notebook braucht. Führen Sie die folgende Zelle deshalb zu Beginn einmal aus, sie installiert die Pakete. Meldet ein späterer Import trotzdem einen Fehler, starten Sie den Kernel neu (**Kernel** → **Restart Kernel**) und führen Sie die Zellen erneut aus.

In [ ]:
%pip install --quiet numpy scipy matplotlib uncertainties odrpack==0.6.1

In diesem Notebook werden die Messdaten zum Versuch IL ausgewertet (Anleitung Kapitel 5, Auswertung in Abschnitt 5.6).

Es werden verwendet:

- `numpy` für Rechnungen mit Messwerten
- `matplotlib.pyplot` für Diagramme
- `odr_fit` aus `odrpack` für den linearen Fit mit Unsicherheiten in beide Richtungen

Die Auswertung besteht aus fünf Teilen:

1. Ohmscher Widerstand: Kennlinie und Widerstand
2. Diode: Kennlinie und Widerstand bei verschiedenen Stromstärken
3. Elektrolyt: Kennlinie und Prüfung auf ohmsches Verhalten
4. Elektrolyt: Einfluss der Geometrie, spezifischer Widerstand und Vergleich mit dem Konduktometer
5. Elektrolyt: Einfluss der Temperatur

Notation wie im Versuch MEDA: $\bar x$ Mittelwert, $s_x$ Standardabweichung, $u_x$ Messunsicherheit. Im Bericht werden Ergebnisse in der Form $(\bar x \pm u_x)\,\mathrm{Einheit}$ angegeben. $u_x$ wird auf eine signifikante Stelle aufgerundet (auf zwei, wenn die erste Stelle eine 1 oder 2 ist), der Bestwert auf dieselbe Dezimalstelle gerundet.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from odrpack import odr_fit

## 1. Ohmscher Widerstand

In diesem Teil wird aus der Kennlinie der Widerstand bestimmt. Es gilt das Ohmsche Gesetz (Gl. 5.5):

$$
U = R\,I
$$

Wird $U$ gegen $I$ aufgetragen, ist die Steigung der Ausgleichsgeraden der Widerstand.

Zuerst werden die Messwerte und ihre Unsicherheiten definiert, danach wird die Kennlinie dargestellt und der Widerstand aus dem Fit bestimmt.

### Messwerte und Unsicherheiten definieren

Hier werden die gemessenen Stromstärken $I$ und Spannungen $U$ definiert.

Die Messungenauigkeiten der Multimeter entnehmen Sie den Datenblättern im Versuchsraum (Abschnitt 5.4.2). Sie werden als Prozentsatz des Messwerts plus eine Anzahl Stellen der letzten Anzeigeziffer angegeben:

$$
u = p \cdot \text{Messwert} + k \cdot \text{letzte Anzeigestelle}
$$

Was eine Anzeigestelle bedeutet, hängt vom eingestellten Messbereich ab, notieren Sie ihn deshalb bei der Messung mit. Abschnitt 5.6 weist ausdrücklich darauf hin, dass nicht nur die Digitalisierung, sondern auch die Messungenauigkeit des Multimeters zu berücksichtigen ist.

In [ ]:
# Ihr Code hier


### Diagramm der Messwerte erstellen

Vor dem Fit werden die Messwerte dargestellt: $U$ gegen $I$ mit Fehlerbalken. Prüfen Sie am Verlauf, ob der lineare Zusammenhang nach Gl. 5.5 vorliegt.

Zur Erinnerung: `plt.errorbar(x, y, xerr=u_x, yerr=u_y, marker="o", linestyle="", label="Messwerte")` zeichnet die Messwerte mit Fehlerbalken.

Achsen werden in der Form „Größe in Einheit“ beschriftet.

In [ ]:
# Ihr Code hier


### Widerstand aus dem linearen Fit bestimmen

Die Ausgleichsgerade hat die Form

$$
U = m\,I + b
$$

Aus dem Vergleich mit Gl. 5.5 folgt $R = m$.

Weil beide Größen Unsicherheiten tragen, wird der Fit mit `odr_fit` durchgeführt. Er gewichtet die Messwerte mit ihren Unsicherheiten in beide Richtungen, genau wie die Methode minimaler und maximaler Steigung aus Abschnitt 1.3.2 die Fehlerbalken in $x$- und $y$-Richtung verwendet.

Zur Erinnerung: ODR berücksichtigt Unsicherheiten in $x$- und $y$-Richtung. Der Fit läuft in vier Schritten.

**1. Modellfunktion definieren.** Die Parameter stehen gemeinsam im Array `beta`. Die Funktion bekommt zuerst `x` und danach `beta`:

```python
def linear_odr(x, beta):
    return beta[0] * x + beta[1]
```

**2. Gewichte berechnen.** `odr_fit` erwartet die Unsicherheiten nicht direkt, sondern als Gewichte. Das Gewicht eines Messwerts ist eins geteilt durch das Quadrat seiner Unsicherheit:

$$
w_x = \frac{1}{u_x^2}
\qquad
w_y = \frac{1}{u_y^2}
$$

**3. Fit durchführen.** Übergeben werden die Modellfunktion, die Messwerte, die Startwerte `beta0` und die Gewichte:

```python
output = odr_fit(linear_odr, x, y, beta0=[1, 0], weight_x=1/u_x**2, weight_y=1/u_y**2)
```

**4. Ergebnisse auslesen.** Die Parameter stehen in `output.beta`. Ihre Unsicherheiten stehen wie bei `curve_fit` auf der Diagonalen der Kovarianzmatrix, hier heißt sie `output.cov_beta`:

```python
u_m = np.sqrt(output.cov_beta[0, 0])   # Steigung
u_b = np.sqrt(output.cov_beta[1, 1])   # Achsenabschnitt
```

**Wichtig:** `beta0` sind die Startwerte. `odr_fit` arbeitet mit relativen Schrittweiten und braucht sie in der richtigen Größenordnung, bei sehr kleinen Zahlen darf kein Startwert genau null sein.

In [ ]:
# Ihr Code hier


## 2. Diode

Abschnitt 5.6 verlangt für die Diode die grafische Darstellung der Kennlinie und die Berechnung des Widerstands bei verschiedenen Stromstärken. Es gilt (Gl. 5.4):

$$
R = \frac{U}{I}
$$

### Messwerte definieren

Hier werden die Stromstärken und Spannungen der Diode für beide Polaritäten definiert.

Die Unsicherheiten folgen wie in Teil 1 aus den Datenblattangaben für den jeweils eingestellten Messbereich.

In [ ]:
# Ihr Code hier


### Kennlinie darstellen

Es wird $U$ gegen $I$ mit Fehlerbalken dargestellt.

In [ ]:
# Ihr Code hier


### Widerstand bei verschiedenen Stromstärken berechnen

Für jeden Messpunkt mit $I \neq 0$ wird der Widerstand nach Gl. 5.4 berechnet.

In [ ]:
# Ihr Code hier


## 3. Elektrolyt: Kennlinie

In diesem Teil wird die Kennlinie der 0,1-molaren KCl-Lösung dargestellt. Abschnitt 5.6 verlangt hier die grafische Darstellung **ohne** Berechnung des Widerstands, aber mit Diskussion.

### Messwerte definieren

Hier werden Stromstärken und Spannungen der Kennlinie definiert, dazu Elektrodenabstand $l$ und Elektrodenfläche $A$, die für Teil 4 gebraucht werden. Die Fläche ist das Produkt aus Füllhöhe und Elektrodenbreite.

Verwenden Sie für die Unsicherheiten die Datenblattangaben, die zu Ihrer Messung passen (Gleich- oder Wechselspannung, Messbereich).

In [ ]:
# Ihr Code hier


### Kennlinie darstellen

Es wird $U$ gegen $I$ mit Fehlerbalken dargestellt.

Diskutieren Sie anhand des Diagramms die Frage aus Abschnitt 5.6: Liegt ein ohmsches Verhalten vor?

In [ ]:
# Ihr Code hier


## 4. Elektrolyt: spezifischer Widerstand

In diesem Teil wird aus Strom, Spannung und den Abmessungen der spezifische Widerstand berechnet. Aus Gl. 5.8 und Gl. 5.4 folgt:

$$
\rho = R\,\frac{A}{l} = \frac{U\,A}{I\,l}
$$

Abschnitt 5.6 verlangt die Berechnung **für jeden einzelnen Messpunkt** inklusive Unsicherheit und den Vergleich mit den Werten des Konduktometers.

Zuerst werden die Messwerte definiert. Danach werden die Widerstände der Messreihen zur Geometrie, der spezifische Widerstand aller Messpunkte und der Wert des Konduktometers berechnet.

### Messwerte der Geometrievariationen definieren

Hier werden die Messpaare für drei verschiedene Elektrodenflächen und drei verschiedene Abstände definiert, jeweils bei einem Strom von etwa $3\,\mathrm{mA}$. Dazu kommt die mit dem Konduktometer gemessene Leitfähigkeit $\sigma$.

In [ ]:
# Ihr Code hier


### Widerstand bei veränderter Fläche und verändertem Abstand berechnen

Berechnen Sie für jedes dieser Messpaare den Widerstand $R$ (Gl. 5.4) bzw. den Leitwert $G$ (Gl. 5.3).

Diskutieren Sie den Trend bzw. die Proportionalität direkt, ohne eine Ausgleichsgerade zu fitten (Abschnitt 5.4.1).

In [ ]:
# Ihr Code hier


### Spezifischen Widerstand berechnen

Berechnen Sie $\rho$ für jedes Messpaar der beiden Messreihen aus Teil 4 (Variation der Fläche und des Abstands). Die Kennlinie aus Teil 3 wird ohne Berechnung des Widerstands ausgewertet (Abschnitt 5.6). Die Unsicherheit folgt aus Gaußscher Fehlerfortpflanzung über $U$, $I$, $A$ und $l$:

$$
u_\rho =
\sqrt{
\left(\frac{\partial \rho}{\partial U}\,u_U\right)^2
+\left(\frac{\partial \rho}{\partial I}\,u_I\right)^2
+\left(\frac{\partial \rho}{\partial A}\,u_A\right)^2
+\left(\frac{\partial \rho}{\partial l}\,u_l\right)^2
}
$$

Geben Sie die Werte in $\Omega\,\mathrm{cm}$ aus, dann lassen sie sich mit dem Konduktometer vergleichen.

**Hinweis:** Die Fehlerfortpflanzung kann auch mit dem Paket `uncertainties` berechnet werden. Dazu werden die Messgrößen zusammen mit ihren Unsicherheiten als Arrays angelegt, zum Beispiel `U = unp.uarray(U_A, u_U_A)` und `A = unp.uarray(A_var, u_A_var)` (nach `import uncertainties.unumpy as unp`). Mit diesen Arrays wird wie gewohnt gerechnet, Funktionen wie Wurzel werden dabei als `unp.sqrt()` geschrieben. Bestwert und Unsicherheit des Ergebnisses liefern `unp.nominal_values(rho)` und `unp.std_devs(rho)`. Achten Sie auf die richtige Angabe signifikanter Stellen. `uncertainties` rundet Unsicherheiten kaufmännisch statt aufzurunden.

In [ ]:
# Ihr Code hier


### Mit dem Konduktometer vergleichen

Das Konduktometer misst die Leitfähigkeit $\sigma$ direkt. Nach Gl. 5.7 ist der spezifische Widerstand ihr Kehrwert:

$$
\rho = \frac{1}{\sigma}
$$

Die Unsicherheit folgt aus Gaußscher Fehlerfortpflanzung über $\sigma$.

Vergleichen Sie den Wert mit den aus Strom und Spannung berechneten und schätzen Sie die Grenzen der Aussagekraft solcher Messungen ab (Abschnitt 5.4.1).

In [ ]:
# Ihr Code hier


## 5. Elektrolyt: Einfluss der Temperatur

Abschnitt 5.6 verlangt nur eine qualitative Diskussion der Temperaturabhängigkeit des Elektrolyten, eine Auswertung im Notebook ist dafür nicht nötig. Grundlage ist Ihre Messung bei erhöhter Temperatur (Abschnitt 5.4.1).

## 6. Zusammenfassung der Ergebnisse

Zum Schluss werden die wichtigsten Ergebnisse des gesamten Versuchs gesammelt ausgegeben.

Die Zusammenfassung enthält:

1. den Widerstand des ohmschen Bauteils,

$$
R \pm u_R
$$

2. den Widerstand der Diode bei verschiedenen Stromstärken,
3. den spezifischen Widerstand des Elektrolyten und den Vergleich mit dem Konduktometer,

$$
\rho \pm u_\rho
$$

Achten Sie darauf, alle Ergebnisse mit passenden Einheiten auszugeben. Denken Sie daran, die Ergebnisse im Bericht mit der richtigen Anzahl signifikanter Stellen anzugeben.

In [ ]:
# Ihr Code hier
